
# Metrology 디버깅: 한 trial 의 이미지로 단계별 실행

운용에서는 `MetrologyRun` 을 스크립트에서 부른다. 여기서는 같은 이미지(fiber 노출 1장 + fiducial 전용 노출 0–2장)로
같은 함수들을 한 단계씩 돌리며 검출 → fiducial 매칭·변환 fit → fiber 매칭 → 각도 json / offset 표 를 확인한다.

| 설정 | 뜻 |
|---|---|
| `BROKEN_FIBERS` | 빛이 안 나는 positioner (분광기에 fiber 가 안 연결된 33 홀, 끊어진 G4 포함). 목표·json 순서는 그대로, 이미지에서 찾지 않음 (결과 NaN, 회전량 0) |
| `FIXED_FIBERS` | 빛은 나오지만 고장나 고정된 positioner (sky fiber 가 꽂힌 G7, H8, J6). 홀 ID 만 알면 된다: 그 홀에서 arm 길이 안이면서 살아 있는 fiber 목표 근처가 아닌 검출을 그 점으로 보고 뺀다 |
| `SKY_FIBERS` | sky fiber 가 꽂힌 홀 (표에서 −9). 위와 같은 규칙으로 뺀다 |
| `ANGLE_SIGN` | 출력 각도 부호. 기본 (1, 1): alpha 반시계 양, beta ≥ 0 |
| `ANGLE_ZERO` | positioner 명령각 영점. 기본 (30, 150): 기하각 (30, 150) 이 명령 (0, 0). 출력·각도 입력 모두 명령각 |


## 촬영 (take_exposure.py)
노출은 `kspec_metrology/notebooks/take_exposure.py` 로 직접 찍는다 (노출시간 [ms]; 이름 끝에 `_0` 이 자동으로 붙는다; `--dry-run` 은 이름만 확인).
```bash
python kspec_metrology/notebooks/take_exposure.py --fiber 400 --fid 2 10 --out /data/tile01_t1
```
→ `tile01_t1_fib_400ms_0.fits`(fiber 노출), `tile01_t1_fid1_2ms_0.fits`, `tile01_t1_fid2_10ms_0.fits`(fiducial 전용 노출 2장). 끝에 찍히는 `FIBER_IMAGE =` / `FID_IMAGES =` 두 줄을 1절에 붙여 넣는다.
fiber 와 fiducial 을 한 노출에 담으면 `--exptime 400 --out /data/tile01_t1.fits` 한 장과 `FID_IMAGES = []`.


In [ ]:

import os, sys, json, glob
PKG_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # 이 노트북이 kspec_metrology/notebooks/ 에 있을 때 저장소 루트
OUT_DIR  = os.path.expanduser('~/metrology_out')                   # <- 결과를 쓸 폴더

# ---- 설정: 바꾸면 커널을 재시작한다 (표와 제외 목록은 import 때 읽힌다) ----
FIBER_TABLE   = None            # None = 패키지 기본 표 (config73). 다른 표면 경로
# 빈 리스트 [] 는 패키지 기본값(analysis/fiber_config.py, 설치표 2026-10-12 기준) 을 그대로 쓴다. 적으면 그것이 우선.
BROKEN_FIBERS = []              # 빛이 안 나는 positioner 홀 ID. 기본: 분광기에 fiber 가 안 연결된 33 홀, 끊어진 G4 포함 (= cfg.FIBER_EXCLUDE)
FIXED_FIBERS  = []              # 켜져 있지만 움직이지 않는 positioner 홀 ID. 기본: ['G7', 'H8', 'J6'] (sky fiber 가 꽂힌 고정 positioner)
SKY_FIBERS    = []              # 표에서 -9 인 홀의 sky fiber. 기본: ['B7', 'H1', 'N7']
ANGLE_SIGN    = (1, 1)          # 출력 각도 = 부호 x 기하각 (alpha 반시계 양, beta >= 0)
ANGLE_ZERO    = (30, 150)       # positioner 명령각 영점: 기하각 (30, 150) = 명령 (0, 0). 출력·각도 입력 모두 명령각 = 부호 x 기하각 - 영점

if FIBER_TABLE:
    os.environ['KSPEC_FIBER_TABLE'] = FIBER_TABLE
os.environ['KSPEC_FIBER_EXCLUDE'] = ','.join(BROKEN_FIBERS)
os.environ['KSPEC_FIBER_FIXED']   = ','.join(FIXED_FIBERS)
os.environ['KSPEC_SKY_FIBERS']    = ','.join(SKY_FIBERS)
os.environ['KSPEC_ANGLE_SIGN']    = f'{ANGLE_SIGN[0]},{ANGLE_SIGN[1]}'
os.environ['KSPEC_ANGLE_ZERO']    = f'{ANGLE_ZERO[0]},{ANGLE_ZERO[1]}'
os.environ['KSPEC_USE_MEASURED']  = '1'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, PKG_DIR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 80
from kspec_metrology import lab
from kspec_metrology.analysis import fiber_config as cfg
print(f'fiber table {cfg.FIBER_TABLE_PATH}: {len(cfg.fiber_ids())} positioners')
print('broken', cfg.FIBER_EXCLUDE, '| fixed', cfg.FIBER_FIXED, '| sky', cfg.SKY_FIBERS, '| angle sign', cfg.ANGLE_SIGN, '| angle zero', cfg.ANGLE_ZERO)


In [ ]:

MAGNIFICATION  = None           # camera -> focal 배율 초기값. None = 패키지 기본 (2026-01 commissioning fit -8.86, 설치된 실카메라). 10/17 시험 셋업 영상은 8.40
MODEL          = 'full+affine'  # 본 장비. fiducial 30개 기준 leave-one-out 이 가장 좋았다 (README 표). 실험실 광학계는 'radial+affine'
NWINDOW        = 40
NITER_RECENTER = 1
THRESHOLD      = 1e3            # 시작 문턱 [ADU, 배경 포함]; 검출 개수에 맞춰 자동으로 올리고 내린다
MIN_DIST       = 80             # 이보다 가까운 검출은 하나로 합침 [px]
FINDER_OPTS    = {'minarea': 9, 'deblend_cont': 1.0}
N_EXTRA        = 0              # FIXED/SKY 외의 여분 밝은 점 (반사광 등)
SAT_LEVEL      = 65000.
TILE           = 'tile01'



## 1. 입력
`object.info` (xp/yp 또는 각도) 와 FITS 경로. fiber 노출에 fiducial 이 같이 찍혀 있어도 되고(교차 마스킹), fiducial 전용 노출이 0–2장이어도 된다.


In [ ]:

TARGET_FILE = '/path/to/object.info'
FIBER_IMAGE = '/path/to/fiber.fits'
FID_IMAGES  = []                     # 예: ['/path/fid_2ms.fits', '/path/fid_10ms.fits']
xp, yp, kind = lab.read_targets(TARGET_FILE)
print(f'targets {xp.size} ({kind})')



## 2. 검출 점검 (문턱값 탐색 없이 한 번)
개수가 기대(살아 있는 fiber + fiducial + 고정/sky 점)와 맞는지, 한 spot 이 여러 peak 으로 쪼개지는지(주황) 본다.


In [ ]:

for f in [FIBER_IMAGE] + list(FID_IMAGES):
    lab.check_peaks(f, threshold=THRESHOLD, min_dist=MIN_DIST, finder_opts=FINDER_OPTS, show_crops=16)



## 3. 측정 (mtlcal 한 번)
`lab.measure` 는 이미지를 패키지 이름 규칙으로 `OUT_DIR/data` 에 링크하고 `mtlcal` 을 부른다. `MetrologyRun.trial(expose=False)` 와 같은 계산이다.


In [ ]:

res = lab.measure(TARGET_FILE, FIBER_IMAGE, FID_IMAGES, out_dir=OUT_DIR, tile=TILE, itrial=1,
                  magnification=MAGNIFICATION, model=MODEL, nwindow=NWINDOW, threshold=THRESHOLD,
                  niter_recenter=NITER_RECENTER, sat_level=SAT_LEVEL, n_extra=N_EXTRA,
                  min_dist=MIN_DIST, finder_opts=FINDER_OPTS)
tab = lab.offsets_table(res); err = np.hypot(tab.dx_um, tab.dy_um)
print(f'fiber - target: median {np.nanmedian(err):.1f} um, max {np.nanmax(err):.1f} um')
print('angles  :', res['json']); print('offsets :', res['offsets'])
lab.show_offsets(res['offsets']);
tab.round(3)



## 4. 단계별 그림
fiducial 세트별 검출(어느 세트를 썼는지), fiber 이미지 매칭, 초점면 잔차. fiducial 잔차 rms 가 수십 µm 를 크게 넘으면 모델·배율·매칭을 의심한다.


In [ ]:

info = lab.show_steps(res)



## 5. 같은 배치로 반복 (positioner 를 옮긴 뒤)
새 이미지로 `itrial=None` 이면 다음 번호가 되고 누적 각도 = 직전 json + 이번 회전량. `lab.history(OUT_DIR, TILE)` 로 trial 별 오차.


In [ ]:

# res2 = lab.measure(TARGET_FILE, '/path/to/fiber_trial2.fits', [], out_dir=OUT_DIR, tile=TILE, itrial=None,
#                    magnification=MAGNIFICATION, model=MODEL, nwindow=NWINDOW, threshold=THRESHOLD,
#                    niter_recenter=NITER_RECENTER, sat_level=SAT_LEVEL, n_extra=N_EXTRA, min_dist=MIN_DIST, finder_opts=FINDER_OPTS)
# lab.history(OUT_DIR, TILE)
